# Ten Contracts Dataset — QA Evaluation Analysis

Comparison of NKG variants, HyperGraphRAG, GraphRAG, and KGGen across all categories.

Scoring is computed by pooling individual question scores across all 10 contracts so every question is weighted equally regardless of document length.


In [25]:
import json
import glob
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np

BASE = Path('/home/taluka/discourse_hyper_graph')


## Registering result directories

`register(path, name, ...)` is the only thing you need to add a new result directory to every table below. `path` is relative to `BASE` and must contain one subdirectory per contract.

Two on-disk layouts are supported automatically:
- **combined**: each contract dir has a single `results.json` with both `qa_evaluation` and `construction` (HyperGraphRAG / GraphRAG / KGGen).
- **split / hashed**: each contract dir has `construction.json` plus one or more `eval_qa_<hash>.json` files (NKG). The hash identifies a specific retrieval-args config run against the same constructed KG.

For the hashed layout:
- If a directory only ever has one distinct hash across all its contracts, `register` picks it up automatically — no arguments needed.
- If a directory has more than one hash, pass `hash="..."` to pick one, or `include_all_hashes=True` to register every hash as its own row (named `"<name> [<hash>]"`).
- **A hash is only ever registered if it is present in every contract subdirectory.** If a hash is missing for even one contract, that registration is skipped and a message is printed — nothing raises and nothing silently averages over a partial set of contracts.


In [26]:
# ── low-level file helpers ───────────────────────────────────────────────────

def _read_json(path):
    with open(path) as f:
        return json.load(f)


def _extract_tokens(metrics):
    """metrics: dict possibly containing llm_prompt_tokens / llm_completion_tokens / llm_total_tokens."""
    if not metrics:
        return None
    total = metrics.get('llm_total_tokens', 0)
    if not total:
        return None
    return {
        'prompt': metrics['llm_prompt_tokens'],
        'completion': metrics['llm_completion_tokens'],
        'total': total,
    }


def _find_contract_dirs(path):
    full = BASE / path
    if not full.exists():
        return []
    return sorted(d for d in full.iterdir() if d.is_dir())


def _parse_eval_hash(eval_file):
    """
    Extract the hash from an eval file name. Supports both the original NKG naming
    (eval_qa_<hash>.json) and the newer search engine's (eval_qa_<engine_tag>_<hash>.json,
    e.g. eval_qa_new_3a60284f5a.json) — the hash is always the last '_'-separated token.
    """
    rest = eval_file.stem[len('eval_qa_'):]
    return rest.rsplit('_', 1)[-1]


def _discover_hashes(contract_dirs):
    """Return (hash -> set of contract dir names that have it, has_combined_results_json)."""
    hash_to_contracts = defaultdict(set)
    has_results_json = False
    for d in contract_dirs:
        if (d / 'results.json').exists():
            has_results_json = True
            continue
        for f in d.glob('eval_qa_*.json'):
            hash_to_contracts[_parse_eval_hash(f)].add(d.name)
    return hash_to_contracts, has_results_json


def _load_contract_record(contract_dir, hash=None):
    """Return {'contract': name, 'qa': qa_evaluation dict or None, 'construction': tokens dict or None}."""
    results_file = contract_dir / 'results.json'
    if results_file.exists():
        d = _read_json(results_file)
        return {
            'contract': contract_dir.name,
            'qa': d.get('qa_evaluation'),
            'construction': _extract_tokens(d.get('construction')),
        }

    candidates = sorted(contract_dir.glob('eval_qa_*.json'))
    if hash is not None:
        candidates = [f for f in candidates if _parse_eval_hash(f) == hash]
    if not candidates:
        return {'contract': contract_dir.name, 'qa': None, 'construction': None}
    eval_file = candidates[0]

    qa = _read_json(eval_file).get('qa_evaluation')
    construction_file = contract_dir / 'construction.json'
    tokens = _extract_tokens(_read_json(construction_file).get('metrics')) if construction_file.exists() else None
    return {'contract': contract_dir.name, 'qa': qa, 'construction': tokens}


In [27]:
# ── the one function you call to add a system to every table below ──────────

SYSTEMS = {}  # display name -> list of per-contract records, in registration order


def register(path, name, hash=None, include_all_hashes=False):
    """
    Register a results directory so it appears in the accuracy / token tables below.

    path:   directory relative to BASE, containing one subdirectory per contract.
    name:   nickname used as the row label in the tables.
    hash:   pick this specific eval hash (only for the split/hashed NKG layout).
    include_all_hashes: register every hash found as its own row "<name> [<hash>]".

    A hash is only registered if every contract subdirectory has it — partial
    coverage is rejected with a printed message, nothing is raised.

    Returns the list of names actually registered (may be empty).
    """
    contract_dirs = _find_contract_dirs(path)
    if not contract_dirs:
        print(f"[{name}] no contract subdirectories found under {path!r}")
        return []

    hash_to_contracts, has_results_json = _discover_hashes(contract_dirs)
    n_total = len(contract_dirs)

    if has_results_json or not hash_to_contracts:
        SYSTEMS[name] = [_load_contract_record(d) for d in contract_dirs]
        return [name]

    def _register_one_hash(h, label):
        n_have = len(hash_to_contracts.get(h, set()))
        if n_have != n_total:
            print(
                f"[{label}] hash '{h}' only found in {n_have}/{n_total} contracts "
                f"under {path!r} — skipping (a hash must be present for every contract)."
            )
            return False
        SYSTEMS[label] = [_load_contract_record(d, hash=h) for d in contract_dirs]
        return True

    if hash is not None:
        return [name] if _register_one_hash(hash, name) else []

    if include_all_hashes:
        return [f"{name} [{h}]" for h in sorted(hash_to_contracts) if _register_one_hash(h, f"{name} [{h}]")]

    if len(hash_to_contracts) == 1:
        (h,) = hash_to_contracts
        return [name] if _register_one_hash(h, name) else []

    print(
        f"[{name}] found {len(hash_to_contracts)} different eval hashes under {path!r}: "
        f"{sorted(hash_to_contracts)}. Pass hash=\'...\' to pick one, or "
        f"include_all_hashes=True to register all of them."
    )
    return []


## Registered systems

Add or edit `register(...)` calls here — everything else in the notebook re-runs off of `SYSTEMS` automatically.


In [28]:
SYSTEMS.clear()

register('latest_results/insurance/nkg/ten_contracts_dataset', 'NKG')
register('latest_results/insurance/hypergraph/ten_contracts_dataset', 'HyperGraphRAG')
register('latest_results/insurance/graphrag_1/ten_contracts_dataset', 'GraphRAG')
register('results/insurance/kggen/ten_contracts_dataset', 'KGGen')

# register(
#     'token_reduction_experiments/insurance/nkg/remove_round_1/ten_contracts_dataset',
#     'NKG remove_round_1',
# )
register(
    'token_reduction_experiments/insurance/nkg/remove_round_1/no_cot_1/ten_contracts_dataset',
    'NKG remove_round_1 (no_cot_1)',
)
# register(
#     'token_reduction_experiments/insurance/nkg/remove_round_1_3/no_cot/ten_contracts_dataset',
#     'NKG remove_round_1_3 (no_cot)',
# )
# register(
#     'token_reduction_experiments/insurance/nkg/remove_round_1_3/no_cot/neutral_desc/ten_contracts_dataset',
#     'NKG remove_round_1_3 (no_cot, neutral)',
# )
# register(
#     'token_reduction_experiments/insurance/nkg/remove_round_1_3/no_cot/neutral_desc/concise_summary/test_remove_cache/ten_contracts_dataset',
#     'NKG rr1_3 (no_cot, neutral, concise, no-cache)',
# )
#
# # clustered_40 / fact_batch_20 — two retrieval variants sharing the same constructed KG,
# # disambiguated by hash.
# _CLUST40_BASE = (
#     'token_reduction_experiments/insurance/nkg/remove_round_1_3/no_cot/neutral_desc/'
#     'concise_summary/test_remove_cache/clustered_40/fact_batch_20/ten_contracts_dataset'
# )
# register(_CLUST40_BASE, 'NKG clust_40 — baseline retrieval', hash='ce859e112a')
# register(_CLUST40_BASE, 'NKG clust_40 — dedup+neutral retrieval', hash='dbe4fe307b')

# Newest variant: no query-type routing + string-based dedup on top of clust_40/fact_batch_20.
register(
    'token_reduction_experiments/insurance/nkg/remove_round_1_3/no_cot/neutral_desc/'
    'concise_summary/remove_cache/clustered_40/fact_batch_20/no_qt/string_dedup/ten_contracts_dataset',
    'NKG rr1_3 (no_qt, string_dedup)',
)

register(
    'token_reduction_experiments/insurance/nkg/remove_round_1_3/no_cot/neutral_desc/'
    'concise_summary/remove_cache/clustered_40/fact_batch_20/yes_qt/string_dedup/ten_contracts_dataset',
    'NKG rr1_3 (yes_qt, string_dedup)',
)

register(
    "token_reduction_experiments/insurance/nkg/remove_round_1/no_cot/neutral_desc/concise_summary/remove_cache/clustered_40/fact_batch_20/yes_qt/string_dedup/ten_contracts_dataset",
    'NKG rr1 (yes_qt, string_dedup)',
    hash="6439de4227"
)

register(
    "token_reduction_experiments/insurance/nkg/remove_round_1/no_cot/neutral_desc/concise_summary/remove_cache/clustered_40/fact_batch_20/yes_qt/string_dedup/ten_contracts_dataset",
    'NKG rr1 (yes_qt, string_dedup) new search',
    hash="3a60284f5a"
)

print(f"\nRegistered {len(SYSTEMS)} systems: {list(SYSTEMS)}")



Registered 9 systems: ['NKG', 'HyperGraphRAG', 'GraphRAG', 'KGGen', 'NKG remove_round_1 (no_cot_1)', 'NKG rr1_3 (no_qt, string_dedup)', 'NKG rr1_3 (yes_qt, string_dedup)', 'NKG rr1 (yes_qt, string_dedup)', 'NKG rr1 (yes_qt, string_dedup) new search']


## QA accuracy — pooled across all 10 contracts


In [29]:
# ── score aggregation ─────────────────────────────────────────────────────────

CATEGORIES = [
    ('CAT1_ATOMIC_LOOKUP',        'Cat 1: Atomic Lookup'),
    ('CAT2_RELATIONAL_LOOKUP',    'Cat 2: Relational Lookup'),
    ('CAT3_CONDITIONAL_REASONING','Cat 3: Conditional Reasoning'),
    ('CAT4_MULTIHOP_REASONING',   'Cat 4: Multi-hop Reasoning'),
    ('CAT5_UNANSWERABLE',         'Cat 5: Unanswerable'),
]
CAT_KEYS = [c[0] for c in CATEGORIES]
DISPLAY_COLS = dict(CATEGORIES)
DISPLAY_COLS['AVG_EXCL_CAT5'] = 'Avg (excl. Cat 5)'
DISPLAY_COLS['AVG_INCL_CAT5'] = 'Avg (incl. Cat 5)'


def pool_scores(records):
    """Pool every individual question score across all contracts. Returns {cat_key: [score, ...]}."""
    pooled = defaultdict(list)
    for r in records:
        qa = r.get('qa')
        if not qa:
            continue
        by_cat = qa.get('by_category', {})
        for cat_key in CAT_KEYS:
            if cat_key in by_cat:
                pooled[cat_key].extend(by_cat[cat_key]['scores'])
    return pooled


def compute_averages(pooled):
    """Per-category average, plus overall averages excluding/including Cat 5, all pooled at the
    question level (so every question counts equally, regardless of which contract or category
    it came from)."""
    row = {}
    excl_scores, incl_scores = [], []
    for cat_key in CAT_KEYS:
        scores = pooled.get(cat_key, [])
        row[cat_key] = np.mean(scores) if scores else np.nan
        incl_scores.extend(scores)
        if cat_key != 'CAT5_UNANSWERABLE':
            excl_scores.extend(scores)
    row['AVG_EXCL_CAT5'] = np.mean(excl_scores) if excl_scores else np.nan
    row['AVG_INCL_CAT5'] = np.mean(incl_scores) if incl_scores else np.nan
    return row


col_order = CAT_KEYS + ['AVG_EXCL_CAT5', 'AVG_INCL_CAT5']
accuracy_df = pd.DataFrame(
    {name: compute_averages(pool_scores(records)) for name, records in SYSTEMS.items()}
).T[col_order]
accuracy_df.index.name = 'System'
accuracy_df


,CAT1_ATOMIC_LOOKUP,CAT2_RELATIONAL_LOOKUP,CAT3_CONDITIONAL_REASONING,CAT4_MULTIHOP_REASONING,CAT5_UNANSWERABLE,AVG_EXCL_CAT5,AVG_INCL_CAT5
System,,,,,,,
NKG,0.975862,0.892857,0.878378,0.842857,0.901961,0.922581,0.919668
HyperGraphRAG,0.782759,0.767857,0.844595,0.857143,0.901961,0.803226,0.817175
GraphRAG,0.962069,0.946429,0.878378,0.857143,0.882353,0.927419,0.921053
KGGen,0.810345,0.669643,0.824324,0.785714,0.892157,0.785484,0.800554
NKG remove_round_1 (no_cot_1),0.975862,0.910714,0.945946,0.914286,0.852941,0.950000,0.936288
"NKG rr1_3 (no_qt, string_dedup)",0.958621,0.830357,0.918919,0.885714,0.921569,0.917742,0.918283
"NKG rr1_3 (yes_qt, string_dedup)",0.968966,0.883929,0.912162,0.842857,0.892157,0.925806,0.921053
"NKG rr1 (yes_qt, string_dedup)",0.975862,0.892857,0.905405,0.857143,0.882353,0.930645,0.923823
"NKG rr1 (yes_qt, string_dedup) new search",0.958621,0.875000,0.918919,0.871429,0.921569,0.924194,0.923823


In [30]:
# ── pretty accuracy table ────────────────────────────────────────────────────

TABLE_STYLES = [
    {'selector': '',
     'props': [('border-collapse', 'collapse'), ('font-family', 'sans-serif'), ('font-size', '13px')]},
    {'selector': 'caption',
     'props': [('font-size', '13px'), ('text-align', 'left'), ('padding-bottom', '8px'),
               ('font-style', 'italic'), ('color', '#222')]},
    {'selector': 'th.col_heading',
     'props': [('background-color', '#2c5f8a'), ('color', '#ffffff'), ('padding', '8px 12px'),
               ('text-align', 'center'), ('border', '1px solid #1e4468'), ('font-weight', 'bold')]},
    {'selector': 'th.row_heading',
     'props': [('background-color', '#e8edf2'), ('color', '#222'), ('padding', '8px 14px'),
               ('text-align', 'left'), ('border', '1px solid #ccd4dc'), ('font-weight', 'bold')]},
    {'selector': 'th.blank',
     'props': [('background-color', '#2c5f8a'), ('border', '1px solid #1e4468')]},
    {'selector': 'td',
     'props': [('background-color', '#ffffff'), ('color', '#222'), ('padding', '7px 12px'),
               ('text-align', 'center'), ('border', '1px solid #ccd4dc')]},
    {'selector': 'tr:nth-child(even) td', 'props': [('background-color', '#edf3fa')]},
    {'selector': 'tr:hover td', 'props': [('background-color', '#d6e8f7')]},
]


def bold_max(s):
    return ['font-weight: bold' if v else '' for v in (s == s.max())]


(
    accuracy_df
    .rename(columns=DISPLAY_COLS)
    .style
    .format('{:.4f}')
    .apply(bold_max, axis=0)
    .set_caption(
        'QA Evaluation — Ten Contracts Dataset · Average scores computed by pooling all '
        'individual question scores across all 10 contracts (each question weighted equally).'
    )
    .set_table_styles(TABLE_STYLES)
)


,Cat 1: Atomic Lookup,Cat 2: Relational Lookup,Cat 3: Conditional Reasoning,Cat 4: Multi-hop Reasoning,Cat 5: Unanswerable,Avg (excl. Cat 5),Avg (incl. Cat 5)
System,,,,,,,
NKG,0.9759,0.8929,0.8784,0.8429,0.9020,0.9226,0.9197
HyperGraphRAG,0.7828,0.7679,0.8446,0.8571,0.9020,0.8032,0.8172
GraphRAG,0.9621,0.9464,0.8784,0.8571,0.8824,0.9274,0.9211
KGGen,0.8103,0.6696,0.8243,0.7857,0.8922,0.7855,0.8006
NKG remove_round_1 (no_cot_1),0.9759,0.9107,0.9459,0.9143,0.8529,0.9500,0.9363
"NKG rr1_3 (no_qt, string_dedup)",0.9586,0.8304,0.9189,0.8857,0.9216,0.9177,0.9183
"NKG rr1_3 (yes_qt, string_dedup)",0.9690,0.8839,0.9122,0.8429,0.8922,0.9258,0.9211
"NKG rr1 (yes_qt, string_dedup)",0.9759,0.8929,0.9054,0.8571,0.8824,0.9306,0.9238
"NKG rr1 (yes_qt, string_dedup) new search",0.9586,0.8750,0.9189,0.8714,0.9216,0.9242,0.9238


In [31]:
# ── question counts per category per system ─────────────────────────────────

count_df = pd.DataFrame({
    name: {cat: len(pool_scores(records).get(cat, [])) for cat in CAT_KEYS}
    for name, records in SYSTEMS.items()
}).T.rename(columns=DISPLAY_COLS)
count_df.index.name = 'System'
count_df.style.set_caption('Question counts per category (pooled across all 10 contracts)')


,Cat 1: Atomic Lookup,Cat 2: Relational Lookup,Cat 3: Conditional Reasoning,Cat 4: Multi-hop Reasoning,Cat 5: Unanswerable
System,,,,,
NKG,145,56,74,35,51
HyperGraphRAG,145,56,74,35,51
GraphRAG,145,56,74,35,51
KGGen,145,56,74,35,51
NKG remove_round_1 (no_cot_1),145,56,74,35,51
"NKG rr1_3 (no_qt, string_dedup)",145,56,74,35,51
"NKG rr1_3 (yes_qt, string_dedup)",145,56,74,35,51
"NKG rr1 (yes_qt, string_dedup)",145,56,74,35,51
"NKG rr1 (yes_qt, string_dedup) new search",145,56,74,35,51


## Per-Contract Breakdown — which system wins on each contract

Two tables below, one per averaging method (incl. / excl. Cat 5). Rows are contracts, columns are
systems, and each cell is that system's own overall average score for that single contract
(straight from its eval_qa file — nothing recomputed here). The bolded cell in each row is the
best-scoring system for that contract.


In [32]:
import re

# ── per-contract breakdown ────────────────────────────────────────────────────
# No recomputation here — each contract's eval_qa file already carries its own
# overall_avg_score_incl_cat5 / overall_avg_score_excl_cat5, we just read them out.
#
# Different systems name their per-contract directories differently (NKG uses the
# bare contract name, e.g. "contract_1_term_life"; others append a run suffix, e.g.
# "contract_1_term_life_qa_fact_20260610_131921"). Canonicalize by stripping any
# trailing "_qa_fact_<timestamp>" so every system's row lines up on the same contract.

def _canonical_contract(contract_dir_name):
    return re.sub(r'_qa_fact_\d+_\d+$', '', contract_dir_name)


def _contract_sort_key(contract_name):
    m = re.search(r'(\d+)', contract_name)
    return (int(m.group(1)) if m else 0, contract_name)


def per_contract_table(key):
    """
    key: 'overall_avg_score_incl_cat5' or 'overall_avg_score_excl_cat5'
    Returns a DataFrame — one row per contract, one column per system — read
    directly from each contract's precomputed qa_evaluation[key].
    """
    rows = {}
    for name, records in SYSTEMS.items():
        for r in records:
            qa = r.get('qa')
            if not qa or key not in qa:
                continue
            contract = _canonical_contract(r['contract'])
            rows.setdefault(contract, {})[name] = qa[key]
    df = pd.DataFrame(rows).T
    df = df.reindex(sorted(df.index, key=_contract_sort_key))
    df.index.name = 'Contract'
    return df


per_contract_incl_df = per_contract_table('overall_avg_score_incl_cat5')
per_contract_excl_df = per_contract_table('overall_avg_score_excl_cat5')


In [33]:
def bold_max_row(row):
    """Bold the highest score in a row (i.e. the best system for that contract)."""
    top = row.max()
    return ['font-weight: bold' if v == top else '' for v in row]


(
    per_contract_excl_df
    .style
    .format('{:.4f}')
    .apply(bold_max_row, axis=1)
    .set_caption(
        'Per-Contract Avg (excl. Cat 5) — bold = best system for that contract.'
    )
    .set_table_styles(TABLE_STYLES)
)


,NKG,HyperGraphRAG,GraphRAG,KGGen,NKG remove_round_1 (no_cot_1),"NKG rr1_3 (no_qt, string_dedup)","NKG rr1_3 (yes_qt, string_dedup)","NKG rr1 (yes_qt, string_dedup)","NKG rr1 (yes_qt, string_dedup) new search"
Contract,,,,,,,,,
contract_1_term_life,1.0000,0.7407,0.9630,0.7593,1.0000,0.9444,0.9815,0.9815,0.9815
contract_2_whole_life,0.9677,0.8710,0.9839,0.9355,0.9839,0.9032,0.9516,0.9355,0.8871
contract_3_universal_life,0.9839,0.7903,0.9839,0.9032,0.9839,0.9839,0.9677,0.9839,0.9839
contract_4_variable_universal,0.9355,0.8387,0.6452,0.8065,0.9355,0.9677,0.9355,0.9677,0.9677
contract_5_final_expense,0.8269,0.7885,0.9038,0.7885,0.8846,0.8846,0.8654,0.8654,0.8654
contract_6_group_term,0.8448,0.8621,0.9138,0.8103,0.9138,0.7931,0.8621,0.8276,0.7931
contract_7_term_with_riders,0.8857,0.8286,0.9429,0.5143,0.9000,0.9429,0.8571,0.9286,0.9000
contract_8_joint_survivorship,0.9219,0.7188,1.0000,0.8125,0.9531,0.8594,0.9219,0.9531,0.9375
contract_9_return_of_premium,0.9062,0.8281,0.9844,0.7812,0.9844,0.9531,0.9531,0.9375,0.9531


In [34]:
(
    per_contract_incl_df
    .style
    .format('{:.4f}')
    .apply(bold_max_row, axis=1)
    .set_caption(
        'Per-Contract Avg (incl. Cat 5) — bold = best system for that contract.'
    )
    .set_table_styles(TABLE_STYLES)
)


,NKG,HyperGraphRAG,GraphRAG,KGGen,NKG remove_round_1 (no_cot_1),"NKG rr1_3 (no_qt, string_dedup)","NKG rr1_3 (yes_qt, string_dedup)","NKG rr1 (yes_qt, string_dedup)","NKG rr1 (yes_qt, string_dedup) new search"
Contract,,,,,,,,,
contract_1_term_life,1.0000,0.7576,0.9545,0.8030,0.9697,0.9545,0.9848,0.9545,0.9848
contract_2_whole_life,0.9722,0.8611,0.9861,0.9167,0.9583,0.9167,0.9306,0.9167,0.8750
contract_3_universal_life,0.9861,0.8194,0.9861,0.8889,0.9861,0.9861,0.9722,0.9861,0.9861
contract_4_variable_universal,0.9167,0.8611,0.6667,0.8333,0.9167,0.9722,0.9444,0.9722,0.9722
contract_5_final_expense,0.8548,0.8226,0.9194,0.8226,0.9032,0.9032,0.8871,0.8871,0.8871
contract_6_group_term,0.8088,0.8235,0.8529,0.7794,0.8676,0.7647,0.8235,0.7941,0.7647
contract_7_term_with_riders,0.9000,0.8500,0.9500,0.5750,0.9125,0.9500,0.8750,0.9375,0.9125
contract_8_joint_survivorship,0.9054,0.7568,0.9730,0.8378,0.9324,0.8514,0.9054,0.9324,0.9189
contract_9_return_of_premium,0.9189,0.8243,0.9865,0.7973,0.9730,0.9595,0.9459,0.9459,0.9595


## Construction Token Usage — Ten Contracts Dataset

Average LLM token usage (prompt, completion, total) per contract during KG construction, and average context tokens fed per question during retrieval.


In [35]:
# ── construction + context token aggregation ─────────────────────────────────

def avg_construction_tokens(records):
    """Mean per-contract construction token usage (each contract weighted equally)."""
    toks = [r['construction'] for r in records if r.get('construction')]
    if not toks:
        return {'Avg Input Tokens': np.nan, 'Avg Completion Tokens': np.nan,
                'Avg Total Tokens': np.nan, 'N Contracts': 0}
    return {
        'Avg Input Tokens':      round(np.mean([t['prompt'] for t in toks])),
        'Avg Completion Tokens': round(np.mean([t['completion'] for t in toks])),
        'Avg Total Tokens':      round(np.mean([t['total'] for t in toks])),
        'N Contracts':           len(toks),
    }


def avg_context_tokens(records):
    """Mean context tokens per question, pooled across all contracts (each question weighted equally)."""
    tokens = [
        r['context_tokens']
        for rec in records if rec.get('qa')
        for r in rec['qa'].get('results', [])
        if r.get('context_tokens') is not None
    ]
    return round(np.mean(tokens)) if tokens else float('nan')


token_rows = {}
for name, records in SYSTEMS.items():
    row = avg_construction_tokens(records)
    row['Avg Context Tokens / Question'] = avg_context_tokens(records)
    token_rows[name] = row

token_df = pd.DataFrame(token_rows).T[
    ['Avg Input Tokens', 'Avg Completion Tokens', 'Avg Total Tokens',
     'Avg Context Tokens / Question', 'N Contracts']
]
token_df.index.name = 'System'
token_df


,Avg Input Tokens,Avg Completion Tokens,Avg Total Tokens,Avg Context Tokens / Question,N Contracts
System,,,,,
NKG,228142,169407,397550,2344,10
HyperGraphRAG,17365,7991,25356,7308,10
GraphRAG,51462,18656,70118,6533,10
KGGen,8197,4802,12999,483,10
NKG remove_round_1 (no_cot_1),74086,43724,117810,2397,10
"NKG rr1_3 (no_qt, string_dedup)",42953,41971,84923,939,10
"NKG rr1_3 (yes_qt, string_dedup)",32748,34963,67711,951,10
"NKG rr1 (yes_qt, string_dedup)",52109,48543,100652,1058,10
"NKG rr1 (yes_qt, string_dedup) new search",52109,48543,100652,880,10


In [36]:
# ── styled construction + context token table ────────────────────────────────

def bold_min_token(s):
    """Bold the minimum value (lower = more efficient)."""
    if s.name == 'N Contracts':
        return ['' for _ in s]
    valid = s.dropna()
    if valid.empty:
        return ['' for _ in s]
    min_val = valid.min()
    return ['font-weight: bold' if (not np.isnan(v) and v == min_val) else '' for v in s]


(
    token_df
    .style
    .format({
        'Avg Input Tokens':              '{:,.0f}',
        'Avg Completion Tokens':         '{:,.0f}',
        'Avg Total Tokens':              '{:,.0f}',
        'Avg Context Tokens / Question': '{:,.0f}',
        'N Contracts':                   '{:.0f}',
    })
    .apply(bold_min_token, axis=0)
    .set_caption(
        'Construction Token Usage — Ten Contracts Dataset · Average LLM tokens per contract '
        'during KG construction and average context tokens fed per question during retrieval. '
        'Bold = lowest (most efficient).'
    )
    .set_table_styles(TABLE_STYLES)
)


,Avg Input Tokens,Avg Completion Tokens,Avg Total Tokens,Avg Context Tokens / Question,N Contracts
System,,,,,
NKG,"228,142","169,407","397,550","2,344",10
HyperGraphRAG,"17,365","7,991","25,356","7,308",10
GraphRAG,"51,462","18,656","70,118","6,533",10
KGGen,"8,197","4,802","12,999",483,10
NKG remove_round_1 (no_cot_1),"74,086","43,724","117,810","2,397",10
"NKG rr1_3 (no_qt, string_dedup)","42,953","41,971","84,923",939,10
"NKG rr1_3 (yes_qt, string_dedup)","32,748","34,963","67,711",951,10
"NKG rr1 (yes_qt, string_dedup)","52,109","48,543","100,652","1,058",10
"NKG rr1 (yes_qt, string_dedup) new search","52,109","48,543","100,652",880,10
